# Laboratorio: Destilar un modelo chico

**Diplomado en Inteligencia Artificial – Universidad de Chile · Módulo IA Generativa · Clase 5**

**Objetivo:** Enseñarle a un modelo chico a resolver un problema como lo resuelve uno más grande, usando un dataset
con las respuestas del grande, y después **ponerlo a disposición**: medir cuánto rinde y hablarle desde un chat.

**Características:**
- Alumno: **Qwen3-0.6B** · Profesor: **Qwen3-4B-Instruct-2507** (misma familia, mismo vocabulario)
- El dataset del profesor viene **ya armado**: 1000 problemas resueltos por él
- Destilación con **LoRA** en la T4, de dos formas: copiando el **texto** o copiando las **probabilidades**
- Un **chat** dentro del notebook para comparar al alumno antes y después
- Sección opcional: una **API compatible con OpenAI** con nuestro modelo

**Duración:** 1,5 horas · **GPU:** T4 (Colab → *Entorno de ejecución → Cambiar tipo de entorno → T4 GPU*)

**¿Qué es Countdown?**
Un juego de números: con **3 números** y las operaciones + − × ÷, hay que llegar a un **número objetivo** usando
cada número exactamente una vez. Por ejemplo, con 44, 19 y 35 llegar a 98: `44 + 19 + 35`. Es fácil de verificar
(se calcula la ecuación) y difícil de adivinar: hay que **buscar**.

**¿Qué es destilar?**
Un modelo grande (el profesor) resuelve muchos problemas. Con sus respuestas entrenamos a un modelo chico (el
alumno) para que se comporte igual. Así se hicieron los DeepSeek-R1-Distill y los modelos chicos de Gemma y Qwen.

**Cómo se usa este notebook:** no hay que escribir código. Los experimentos tienen **perillas** (deslizadores y
listas a la derecha de la celda). Antes de ejecutar, predigan qué va a pasar; después, expliquen lo que vieron.
No se entrega nada.

| Sección | Tema | Tiempo |
|---|---|---|
| 1–4 | Setup, dataset y modelos | 10 min |
| 5 | Alumno contra profesor | 10 min |
| 6 | Destilar | 25 min |
| 7 | Antes contra después | 15 min |
| 8 | Servir: cuánto pesa y cuánto rinde | 10 min |
| 9 | Un chat en el notebook | 15 min |
| 10 | (opcional) Una API compatible con OpenAI | — |

## 1. Setup e instalación
Instala las librerías (~1 min). Si Colab pide reiniciar la sesión, reinicien y sigan desde la sección 2.

In [ ]:
#@title Instalar (ejecutar)
!pip install -q -U transformers accelerate peft bitsandbytes gradio

## 2. Verificar GPU y cargar utilidades
La segunda celda prepara funciones auxiliares (no hace falta leerla).

In [ ]:
#@title Verificar GPU (ejecutar)
import torch

if torch.cuda.is_available():
    print("GPU detectada:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No hay GPU. Cambien el entorno a T4 antes de seguir.")

In [ ]:
#@title Utilidades (ejecutar)
import os

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import contextlib
import gc
import html
import json
import math
import random
import threading
import time
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import torch.nn.functional as F
from IPython.display import HTML, display
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer

import re

INSTRUCCION = ("Using the numbers {nums}, create an equation that equals {target}. You can use basic arithmetic "
               "operations (+, -, *, /) and each number must be used exactly once. Think step by step briefly, then "
               "write the final equation inside <answer> </answer> tags, for example <answer> (1 + 2) / 3 </answer>.")

_PERMITIDOS = re.compile(r"^[\d+\-*/().\s]+$")


def armar_prompt(p):
    return INSTRUCCION.format(nums=p["nums"], target=p["target"])


def mensajes(p):
    return [{"role": "user", "content": armar_prompt(p)}]


def extraer_respuesta(texto):
    """La última ecuación entre <answer> y </answer>, o None si no hay."""
    encontradas = re.findall(r"<answer>(.*?)</answer>", texto, re.S)
    return encontradas[-1].strip() if encontradas else None


def verificar(texto, p):
    """Tres notas, de menos a más exigente:
    formato: hay una respuesta entre <answer> tags;
    reglas:  la ecuación solo usa + - * / ( ) y cada número dado exactamente una vez;
    correcta: además, da el número objetivo."""
    ecuacion = extraer_respuesta(texto)
    nota = {"formato": ecuacion is not None, "reglas": False, "correcta": False, "ecuacion": ecuacion}
    if not ecuacion or not _PERMITIDOS.match(ecuacion) or "**" in ecuacion or "//" in ecuacion:
        return nota
    usados = sorted(int(n) for n in re.findall(r"\d+", ecuacion))
    if usados != sorted(p["nums"]):
        return nota
    nota["reglas"] = True
    try:
        valor = eval(ecuacion, {"__builtins__": None}, {})  # solo dígitos, operadores y paréntesis (regex de arriba)
    except Exception:
        return nota
    nota["correcta"] = abs(float(valor) - p["target"]) < 1e-5
    return nota


def plantilla(tok, msgs):
    """Plantilla de chat de Qwen3 sin modo thinking (si la plantilla no acepta el argumento, sin él)."""
    try:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
URL_DATOS = "https://raw.githubusercontent.com/valbarriere/CC6XXX-Generative-AI/main/Labs/data/countdown_precalculo.json"
MODELO_ALUMNO = "Qwen/Qwen3-0.6B"
MODELO_PROFESOR = "Qwen/Qwen3-4B-Instruct-2507"
MAX_TOKENS = 768
GPU = threading.Lock()   # una generación a la vez (el chat y la API comparten el modelo)
RESULTADOS = {}          # se van guardando los resultados de cada destilación
COLORES = {"alumno antes": "#9aa5b1", "alumno destilado": "#13803b", "profesor (4B)": "#23383B"}


def memoria_gpu():
    return f"{torch.cuda.memory_allocated() / 1e9:.1f} GB" if device == "cuda" else "(sin GPU)"


def liberar():
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()


def autocast():
    return torch.autocast("cuda", dtype=torch.float16) if device == "cuda" else contextlib.nullcontext()


def cargar_tokenizer(nombre):
    t = AutoTokenizer.from_pretrained(nombre)
    t.padding_side = "left"
    if t.pad_token is None:
        t.pad_token = t.eos_token
    return t


def version(modelo, destilado):
    """El alumno original es el mismo modelo con el adaptador LoRA apagado."""
    if modelo is alumno and not destilado and isinstance(alumno, PeftModel):
        return alumno.disable_adapter()
    return contextlib.nullcontext()


@torch.no_grad()
def generar(modelo, tok_, problemas, destilado=True, max_tokens=MAX_TOKENS, lote=32, mostrar=False):
    """Respuesta greedy para cada problema: devuelve filas con el texto, la nota y los tokens."""
    modelo.eval()
    filas = []
    with GPU, version(modelo, destilado):
        for i in range(0, len(problemas), lote):
            grupo = problemas[i:i + lote]
            ids = tok_([plantilla(tok_, mensajes(p)) for p in grupo], return_tensors="pt", padding=True,
                       add_special_tokens=False).to(device)
            with autocast():
                out = modelo.generate(**ids, max_new_tokens=max_tokens, do_sample=False, pad_token_id=tok_.pad_token_id)
            for p, fila in zip(grupo, out[:, ids["input_ids"].shape[1]:]):
                texto = tok_.decode(fila, skip_special_tokens=True)
                n = int((fila != tok_.pad_token_id).sum())
                nota = verificar(texto, p)
                filas.append({"texto": texto, "ntok": n, "cortada": n >= max_tokens,
                              **{c: nota[c] for c in ("formato", "reglas", "correcta")}})
            if mostrar:
                print(f"  {min(i + lote, len(problemas))}/{len(problemas)}", end="\r")
    return filas


def resumen(filas):
    return {"resuelve (%)": 100 * np.mean([f["correcta"] for f in filas]),
            "respeta formato y reglas (%)": 100 * np.mean([f["reglas"] for f in filas]),
            "tokens por respuesta": np.mean([f["ntok"] for f in filas]),
            "se corta sin terminar (%)": 100 * np.mean([f["cortada"] for f in filas])}


def graficar(tabla, titulo=""):
    metricas = list(next(iter(tabla.values())))
    fig, ejes = plt.subplots(1, len(metricas), figsize=(3.4 * len(metricas), 3.4))
    for eje, m in zip(ejes, metricas):
        barras = eje.bar(list(tabla), [tabla[k][m] for k in tabla],
                         color=[COLORES.get(k, "#13803b") for k in tabla])
        for b in barras:
            eje.text(b.get_x() + b.get_width() / 2, b.get_height(), f"{b.get_height():.0f}", ha="center", va="bottom")
        eje.set_title(m, fontsize=9); eje.tick_params(axis="x", labelsize=7, rotation=15)
    fig.suptitle(titulo); plt.tight_layout(); plt.show()


def mostrar_respuesta(titulo, fila):
    color = "#13803b" if fila["correcta"] else ("#d9822b" if fila["reglas"] else "#c2185b")
    estado = ("correcta" if fila["correcta"] else "respeta las reglas pero no llega al objetivo" if fila["reglas"]
              else "se cortó sin terminar" if fila["cortada"] else "no respeta el formato o las reglas")
    display(HTML(f"<div style='margin:6px 0'><b>{html.escape(titulo)}</b> · {fila['ntok']} tokens · "
                 f"<span style='color:{color}'><b>{estado}</b></span><pre style='background:#f6f6f6;color:#1f1f1f;font-size:13px;padding:8px;"
                 f"border-left:4px solid {color};white-space:pre-wrap;max-height:320px;overflow:auto'>"
                 f"{html.escape(fila['texto'])}</pre></div>"))


print("Listo · dispositivo:", device)

## 3. Descargar el dataset del profesor

El profesor (**Qwen3-4B-Instruct-2507**) ya resolvió **1000 problemas** de Countdown con 3 números. Esto lo
corrimos **aparte**, en una T4, y tomó **cerca de una hora**; por eso llegamos con el dataset armado. Si quieren
generarlo ustedes, les compartimos el notebook con que lo hicimos (`Labs/precalculo_countdown.ipynb` en el repo del curso). Para cada problema tenemos
**todo lo que escribió**: su razonamiento y su respuesta final entre `<answer>` y `</answer>`.

También vienen **100 problemas de evaluación**, que el alumno nunca va a ver al entrenar, con las respuestas del
profesor y del alumno ya calculadas en esa misma corrida.

In [ ]:
#@title Descargar el dataset (ejecutar)
ARCHIVO = "countdown_precalculo.json"
if not os.path.exists(ARCHIVO):
    try:
        urllib.request.urlretrieve(URL_DATOS, ARCHIVO)
    except Exception as e:
        print("No pude descargar el dataset:", e)
        print("Súbanlo a mano: panel de archivos (izquierda) → subir → countdown_precalculo.json")
DATOS = json.load(open(ARCHIVO))
ENT, EVA, GEN, META = DATOS["entrenamiento"], DATOS["evaluacion"], DATOS["generalizacion"], DATOS["meta"]
print(f"Profesor: {META['modelo_profesor']} · Alumno: {META['modelo_alumno']}")
print(f"{len(ENT)} problemas para entrenar · {len(EVA)} para evaluar")
print(f"Respuestas del profesor que son correctas: {100 * np.mean([p['profesor']['correcta'] for p in ENT]):.0f}%")
print(f"Largo promedio de sus respuestas: {np.mean([p['profesor']['ntok'] for p in ENT]):.0f} tokens")
print("\n--- Lo que ve el modelo (el prompt) ---\n")
print(armar_prompt(ENT[0]))

In [ ]:
#@title Perillas: mirar una respuesta del profesor
cual = 0  #@param {type:"slider", min:0, max:30, step:1}
p = ENT[cual]
print(f"Números: {p['nums']} · objetivo: {p['target']}\n")
mostrar_respuesta("Profesor", p["profesor"])

### Observaciones
- ¿Cómo busca el profesor? ¿Prueba al azar o tiene un método?
- Lo que vamos a destilar no es solo la respuesta: es **esa forma de buscar**.

## 4. Cargar el alumno y el profesor
El alumno se carga en 32 bits para poder entrenarlo (~2,4 GB). El profesor ocuparía 8 GB en 16 bits; lo cargamos en
**4 bits** (~2,8 GB) porque lo vamos a necesitar en la sección 6. Toma unos minutos (descarga ~9 GB).

In [ ]:
#@title Cargar los dos modelos (ejecutar)
tok = cargar_tokenizer(MODELO_ALUMNO)
alumno = AutoModelForCausalLM.from_pretrained(MODELO_ALUMNO, dtype=torch.float32).to(device).eval()
N_PARAMETROS = sum(p.numel() for p in alumno.parameters())
print(f"Alumno: {N_PARAMETROS / 1e6:.0f} M parámetros · GPU: {memoria_gpu()}")

tok_prof = cargar_tokenizer(MODELO_PROFESOR)
if device == "cuda":
    from transformers import BitsAndBytesConfig
    q4 = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    profesor = AutoModelForCausalLM.from_pretrained(MODELO_PROFESOR, quantization_config=q4, device_map="cuda:0")
else:
    profesor = AutoModelForCausalLM.from_pretrained(MODELO_PROFESOR, dtype=torch.float32)
profesor.eval()
print(f"Profesor cargado · GPU: {memoria_gpu()}")
print("¿Mismo vocabulario?", len(tok) == len(tok_prof) and tok.get_vocab() == tok_prof.get_vocab())

## 5. SECCIÓN 1: Alumno contra profesor

**Objetivo:** Ver en qué se diferencian, para saber qué tiene que aprender el alumno.

Elijan un problema del conjunto de evaluación o **inventen uno** (escriban 3 números y un objetivo). Los dos
modelos responden sin muestreo (siempre lo mismo). **Predigan:** ¿quién acierta? ¿Quién escribe más?

In [ ]:
#@title Perillas: un problema, dos modelos
problema = 0  #@param {type:"slider", min:0, max:99, step:1}
inventar = False  #@param {type:"boolean"}
numeros_propios = "25, 4, 9"  #@param {type:"string"}
objetivo_propio = 91  #@param {type:"integer"}
import re

p = ({"nums": [int(n) for n in re.findall(r"\d+", numeros_propios)], "target": int(objetivo_propio)} if inventar
     else EVA[problema])
print(f"Números: {p['nums']} · objetivo: {p['target']}\n")
for nombre, modelo_, tok_ in [("Alumno (0,6B)", alumno, tok), ("Profesor (4B, en 4 bits)", profesor, tok_prof)]:
    t0 = time.time()
    (fila,) = generar(modelo_, tok_, [p], destilado=False, lote=1)
    mostrar_respuesta(f"{nombre} · {fila['ntok'] / (time.time() - t0):.0f} tokens/s", fila)

Ahora el examen completo: los 100 problemas de evaluación. Las respuestas ya están calculadas (vienen en el dataset,
de la corrida aparte), así que es instantáneo. En vivo, solo el profesor tardaría más de 10 minutos.

In [ ]:
#@title Examen: alumno contra profesor (ejecutar)
ANTES = [p["alumno"] for p in EVA]
PROFESOR_EVA = [p["profesor"] for p in EVA]
graficar({"alumno antes": resumen(ANTES), "profesor (4B)": resumen(PROFESOR_EVA)}, "100 problemas de evaluación")

### Observaciones
- ¿Cuánta diferencia hay en problemas resueltos? ¿Y en largo de las respuestas?
- ¿Qué significa que el alumno se corte sin terminar tantas veces? Miren una de sus respuestas largas arriba: ¿qué
  hace cuando no encuentra la solución?

## 6. SECCIÓN 2: Destilar

**Objetivo:** Entrenar al alumno con las respuestas del profesor.

Usamos **LoRA**: se agregan matrices chicas al lado de cada capa y solo esas se entrenan. Cada vez que ejecuten esta
celda, el entrenamiento parte del alumno **original**. Hay dos decisiones:

**¿Qué se copia?**
- **razonamiento completo:** todo lo que escribió el profesor, la búsqueda y la respuesta;
- **solo la respuesta:** únicamente `<answer> ... </answer>`, sin el razonamiento.

**¿Cómo se copia?**
- **texto (SFT):** el alumno aprende a predecir el token que escribió el profesor. Es lo que hicieron los R1-Distill.
- **probabilidades (logit KD):** en cada posición, el alumno aprende la **distribución** del profesor sobre el
  siguiente token (sus 20 opciones más probables), no solo la que eligió. Para eso el profesor tiene que estar
  cargado: lee cada ejemplo y entrega sus probabilidades.

$$\text{SFT: } -\log p_{\text{alumno}}(y_t) \qquad\qquad \text{KD: } -\sum_{v} p_{\text{profesor}}(v)\,\log p_{\text{alumno}}(v)$$

**Predigan:** ¿cuál de las cuatro combinaciones funciona mejor? ¿Basta con mostrarle la respuesta?

Empiecen con **razonamiento completo + texto** y los valores por defecto. Después prueben las otras (sección 7.3).

In [ ]:
#@title Perillas: destilar
que_se_copia = "razonamiento completo"  #@param ["razonamiento completo", "solo la respuesta"]
como_se_copia = "texto (SFT)"  #@param ["texto (SFT)", "probabilidades (logit KD)"]
ejemplos = 600  #@param {type:"slider", min:100, max:1000, step:50}
epocas = 1  #@param {type:"slider", min:1, max:3, step:1}

FIN = "<|im_end|>"
TOP_K = 20


def clave(p):
    return p["target"], tuple(sorted(p["nums"]))


EN_EVALUACION = {clave(p) for p in EVA}


def armar_ejemplos(que, n):
    datos = []
    for p in ENT:
        r = p["profesor"]
        if not r["correcta"] or clave(p) in EN_EVALUACION:   # solo respuestas aprobadas; nada que esté en el examen
            continue
        texto = r["texto"].strip() if que == "razonamiento completo" else f"<answer> {extraer_respuesta(r['texto'])} </answer>"
        respuesta = tok(texto + FIN, add_special_tokens=False)["input_ids"][:700]   # el 90% de las respuestas tiene menos de 600 tokens
        e = {"respuesta": respuesta,
             "prompt": tok(plantilla(tok, mensajes(p)), add_special_tokens=False)["input_ids"],
             "prompt_prof": tok_prof(plantilla(tok_prof, mensajes(p)), add_special_tokens=False)["input_ids"]}
        datos.append(e)
    random.Random(0).shuffle(datos)
    return datos[:n]


def perdida_ejemplo(e, como):
    """Suma de la pérdida sobre los tokens de la respuesta (el prompt no cuenta). Un ejemplo a la vez: el vocabulario
    tiene 152 mil tokens y las probabilidades de varias respuestas juntas no caben en la T4."""
    a, n = len(e["prompt"]), len(e["respuesta"])
    ids = torch.tensor([e["prompt"] + e["respuesta"]], device=device)
    with autocast():   # solo las posiciones que predicen la respuesta (de a-1 a a+n-2)
        logits = alumno(input_ids=ids, logits_to_keep=n + 1).logits[0, :n]
    lp_alumno = logits.float().log_softmax(-1)
    if como == "texto (SFT)":
        return -lp_alumno.gather(-1, torch.tensor(e["respuesta"], device=device)[:, None]).sum()
    ids_p = torch.tensor([e["prompt_prof"] + e["respuesta"]], device=device)
    with torch.no_grad(), autocast():
        logits_p = profesor(input_ids=ids_p, logits_to_keep=n + 1).logits[0, :n]
    top = logits_p.float().topk(TOP_K, dim=-1)
    p_prof = top.values.softmax(-1)                                      # las 20 opciones del profesor
    return -(p_prof * lp_alumno.gather(-1, top.indices)).sum()


def entrenar(datos, como, epocas, lr=2e-4, rango=16, lote=8):
    global alumno
    if isinstance(alumno, PeftModel):            # cada entrenamiento parte del alumno original
        alumno = alumno.unload()
    alumno = get_peft_model(alumno, LoraConfig(r=rango, lora_alpha=2 * rango, lora_dropout=0.05,
                                               target_modules="all-linear", task_type="CAUSAL_LM"))
    entrenables = [p for p in alumno.parameters() if p.requires_grad]
    print(f"Se entrenan {sum(p.numel() for p in entrenables) / 1e6:.1f} M de {N_PARAMETROS / 1e6:.0f} M parámetros "
          f"({100 * sum(p.numel() for p in entrenables) / N_PARAMETROS:.1f}%)")
    pasos = epocas * math.ceil(len(datos) / lote)
    opt = torch.optim.AdamW(entrenables, lr=lr)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / max(1, pasos // 10)) * max(0.0, 1 - s / pasos))
    escala = torch.amp.GradScaler("cuda", enabled=device == "cuda")
    alumno.train()
    perdidas, t0 = [], time.time()
    for ep in range(epocas):
        random.shuffle(datos)
        for i in range(0, len(datos), lote):
            grupo = datos[i:i + lote]
            n_tokens = sum(len(e["respuesta"]) for e in grupo)
            suma = 0.0
            for e in grupo:                       # se acumulan los gradientes del lote, ejemplo por ejemplo
                s = perdida_ejemplo(e, como)
                escala.scale(s / n_tokens).backward()
                suma += s.item()
            escala.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(entrenables, 1.0)
            escala.step(opt); escala.update(); opt.zero_grad(set_to_none=True); sched.step()
            perdidas.append(suma / n_tokens)
            if len(perdidas) % 10 == 0 or len(perdidas) == pasos:
                falta = (time.time() - t0) / len(perdidas) * (pasos - len(perdidas))
                print(f"  paso {len(perdidas)}/{pasos} · pérdida {np.mean(perdidas[-10:]):.3f} · "
                      f"{time.time() - t0:.0f} s (faltan ~{falta / 60:.1f} min)", flush=True)
    alumno.eval()
    return perdidas


DATOS_SFT = armar_ejemplos(que_se_copia, ejemplos)
print(f"{len(DATOS_SFT)} ejemplos · {np.mean([len(e['respuesta']) for e in DATOS_SFT]):.0f} tokens de respuesta en "
      f"promedio · {como_se_copia}")
PERDIDAS = entrenar(DATOS_SFT, como_se_copia, epocas)
ETIQUETA = f"{que_se_copia} · {como_se_copia} · {len(DATOS_SFT)} ej. · {epocas} ép."
liberar()
plt.figure(figsize=(7, 3.2))
plt.plot(PERDIDAS, alpha=.4)
if len(PERDIDAS) > 10:
    plt.plot(np.convolve(PERDIDAS, np.ones(10) / 10, mode="valid"), lw=2)
plt.xlabel("paso"); plt.ylabel("pérdida"); plt.title(ETIQUETA, fontsize=9); plt.grid(alpha=.3)
plt.show()

### Observaciones
- ¿Qué porcentaje de los parámetros se entrenó? ¿Por qué alcanza con tan poco?
- Que baje la pérdida dice que el alumno **imita mejor** al profesor. ¿Significa que resuelve mejor? Lo vemos ahora.
- Las pérdidas de SFT y de KD no se comparan directamente: miden cosas distintas.

## 7. SECCIÓN 3: Antes contra después

### 7.1 El examen
**Objetivo:** Medir qué aprendió el alumno con los mismos 100 problemas (que nunca vio). Toma 1–3 minutos.

In [ ]:
#@title Examen del alumno destilado (ejecutar)
t0 = time.time()
DESPUES = generar(alumno, tok, EVA, destilado=True, mostrar=True)
print(f"\n100 problemas en {time.time() - t0:.0f} s")
RESULTADOS[ETIQUETA] = resumen(DESPUES)
graficar({"alumno antes": resumen(ANTES), "alumno destilado": resumen(DESPUES), "profesor (4B)": resumen(PROFESOR_EVA)},
         ETIQUETA)
mejoro = [i for i in range(len(EVA)) if DESPUES[i]["correcta"] and not ANTES[i]["correcta"]]
empeoro = [i for i in range(len(EVA)) if ANTES[i]["correcta"] and not DESPUES[i]["correcta"]]
print(f"Ahora resuelve y antes no: {len(mejoro)} · antes resolvía y ahora no: {len(empeoro)}")

In [ ]:
#@title Perillas: mirar un caso
tipo = "ahora lo resuelve"  #@param ["ahora lo resuelve", "antes lo resolvía y ahora no", "cualquiera"]
caso = 0  #@param {type:"slider", min:0, max:30, step:1}

lista = {"ahora lo resuelve": mejoro, "antes lo resolvía y ahora no": empeoro,
         "cualquiera": list(range(len(EVA)))}[tipo]
if not lista:
    print("No hay problemas de ese tipo.")
else:
    i = lista[caso % len(lista)]
    print(f"Números: {EVA[i]['nums']} · objetivo: {EVA[i]['target']}\n")
    mostrar_respuesta("Alumno antes", ANTES[i])
    mostrar_respuesta("Alumno destilado", DESPUES[i])

### Observaciones
- ¿Qué cambió más: resolver, respetar las reglas o el largo? ¿Sigue cortándose sin terminar?
- Comparen el razonamiento del destilado con el del profesor (sección 3). ¿Se parece?

### 7.2 ¿Y con 4 números?
El alumno solo vio problemas de 3 números. ¿Lo que aprendió sirve para un problema **más difícil**? (Como
referencia, el profesor resolvió cerca del 37% de estos en nuestra medición: también le cuestan.)

In [ ]:
#@title Probar con 4 números (ejecutar, ~1 min)
GEN_DESPUES = generar(alumno, tok, GEN, destilado=True, mostrar=True)
graficar({"alumno antes": resumen([p["alumno"] for p in GEN]), "alumno destilado": resumen(GEN_DESPUES)},
         f"{len(GEN)} problemas de 4 números")

### 7.3 Comparar destilaciones
Vuelvan a la **sección 6**, cambien una perilla y ejecuten de nuevo la 6 y la 7.1. Los resultados se acumulan y la
celda siguiente los compara. Repártanse en el grupo:
- **solo la respuesta** contra **razonamiento completo**: ¿basta con ver la respuesta correcta?
- **probabilidades** contra **texto**: ¿sirve más ver todas las opciones del profesor?
- **menos ejemplos** (100, 300): ¿cuántos hacen falta?

In [ ]:
#@title Comparar todas las destilaciones (ejecutar)
nombres = ["alumno antes"] + list(RESULTADOS) + ["profesor (4B)"]
valores = [resumen(ANTES)["resuelve (%)"]] + [r["resuelve (%)"] for r in RESULTADOS.values()] + [resumen(PROFESOR_EVA)["resuelve (%)"]]
plt.figure(figsize=(9, 0.55 * len(nombres) + 1.5))
plt.barh(range(len(nombres)), valores, color=["#9aa5b1"] + ["#13803b"] * len(RESULTADOS) + ["#23383B"])
for i, v in enumerate(valores):
    plt.text(v + 0.5, i, f"{v:.0f}%", va="center")
plt.yticks(range(len(nombres)), nombres, fontsize=8); plt.gca().invert_yaxis()
plt.xlabel("% de problemas resueltos"); plt.xlim(0, 100); plt.title("¿Qué destilación funciona mejor?")
plt.show()

### Observaciones
- ¿Qué pasó con **solo la respuesta**? ¿El alumno aprendió el formato? ¿Aprendió a resolver? ¿Por qué?
- ¿Cuánto ganó **KD** sobre **SFT**, y cuánto más costó (el profesor tiene que leer cada ejemplo)?
- Las empresas entrenan modelos chicos con millones de respuestas de modelos grandes. Con lo que vieron hoy, ¿qué
  cuidarían al armar esos datos?

## 8. SECCIÓN 4: Servir el modelo

**Objetivo:** Medir lo que importa al ponerlo a disposición: cuánto pesa y cuánto rinde.

Primero liberamos al profesor: desde aquí solo atendemos con el alumno. (Si después quieren volver a destilar con
probabilidades, ejecuten de nuevo la sección 4.)

In [ ]:
#@title Liberar al profesor y guardar lo aprendido (ejecutar)
if "profesor" in globals():
    del profesor
liberar()
print("GPU en uso:", memoria_gpu())
alumno.save_pretrained("adaptador_destilado")
peso = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk("adaptador_destilado") for f in fs)
print(f"\nLo aprendido (adaptador LoRA): {peso / 1e6:8.1f} MB")
print(f"Alumno completo en 16 bits:    {N_PARAMETROS * 2 / 1e6:8.1f} MB")
print(f"Profesor en 16 bits:           {4.0e9 * 2 / 1e6:8.1f} MB (aprox.)")

### 8.1 ¿Cuánto rinde?
Pedimos respuestas de 128 tokens a **varios usuarios a la vez** y medimos:
- **tiempo hasta el primer token:** lo que espera el usuario antes de ver algo (el *prefill*);
- **tokens/s por usuario:** la velocidad a la que cada uno ve aparecer el texto;
- **tokens/s en total:** cuánto trabajo saca la GPU.

**Predigan:** si pasamos de 1 a 16 usuarios a la vez, ¿cada usuario va 16 veces más lento? (Lámina "dos máquinas,
dos cuellos de botella".)

In [ ]:
#@title Perillas: medir el rendimiento
usuarios_max = 32  #@param [8, 16, 32, 64]
tokens_respuesta = 128  #@param [64, 128, 256]


@torch.no_grad()
def medir(usuarios, n_tokens):
    ids = tok([plantilla(tok, mensajes(p)) for p in (EVA * 2)[:usuarios]], return_tensors="pt", padding=True,
              add_special_tokens=False).to(device)
    sync = torch.cuda.synchronize if device == "cuda" else (lambda: None)
    with GPU, autocast():
        sync(); t0 = time.time()
        alumno.generate(**ids, max_new_tokens=1, do_sample=False, pad_token_id=tok.pad_token_id)
        sync(); primero = time.time() - t0
        t0 = time.time()
        alumno.generate(**ids, min_new_tokens=n_tokens, max_new_tokens=n_tokens, do_sample=False,
                        pad_token_id=tok.pad_token_id)
        sync(); total = time.time() - t0
    return primero, n_tokens / total, usuarios * n_tokens / total


if device == "cuda":
    torch.cuda.reset_peak_memory_stats()
medir(1, 8)   # calentar
lotes = [u for u in [1, 2, 4, 8, 16, 32, 64] if u <= usuarios_max]
filas = [medir(u, tokens_respuesta) for u in lotes]
print(f"{'usuarios':>8} {'primer token':>13} {'tok/s por usuario':>18} {'tok/s total':>12}")
for u, (pt, por_usuario, total) in zip(lotes, filas):
    print(f"{u:8d} {pt * 1000:10.0f} ms {por_usuario:18.1f} {total:12.1f}")
if device == "cuda":
    print(f"\nMemoria máxima usada: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
fig, ejes = plt.subplots(1, 2, figsize=(11, 3.5))
ejes[0].plot(lotes, [f[2] for f in filas], "o-", color="#13803b"); ejes[0].set_title("tokens/s en total (la GPU)")
ejes[1].plot(lotes, [f[1] for f in filas], "o-", color="#c2185b"); ejes[1].set_title("tokens/s por usuario")
for e in ejes:
    e.set_xlabel("usuarios a la vez"); e.set_xscale("log", base=2); e.grid(alpha=.3); e.set_ylim(bottom=0)
plt.tight_layout(); plt.show()

### Observaciones
- ¿Cuánto bajó la velocidad de cada usuario al pasar de 1 a 16? ¿Y cuánto subió el total?
- ¿Por qué atender a varios a la vez sale casi gratis? (El decode está limitado por **leer los pesos**: leerlos una
  vez sirve para todos los del lote.) Eso es lo que hacen vLLM y los proveedores.
- El alumno destilado escribe menos que el original. ¿Cuánto ahorra eso al servir un millón de preguntas?

## 9. SECCIÓN 5: Un chat en el notebook

**Objetivo:** Usar el modelo como lo usaría una persona.

La celda abre una aplicación con dos pestañas:
- **Countdown:** escriban números y un objetivo; responden el alumno original y el destilado, lado a lado, con la
  velocidad de cada uno;
- **Chat:** una conversación libre con el alumno destilado.

Colab entrega también un **enlace público** (dura unas horas) para abrirlo desde el celular. Cualquiera con el enlace
puede usar su GPU mientras el notebook esté abierto.

Ideas: un problema imposible, uno con números grandes, y en el chat una pregunta que **no** sea de Countdown.

In [ ]:
#@title Abrir el chat (ejecutar)
import gradio as gr


def a_mensajes(historial):
    """El historial de Gradio (pares o mensajes, según la versión) como mensajes de chat."""
    msgs = []
    for h in historial or []:
        if isinstance(h, dict):
            c = h.get("content")
            msgs.append({"role": h["role"], "content": c if isinstance(c, str) else str(c)})
        else:
            msgs += [{"role": "user", "content": h[0]}, {"role": "assistant", "content": h[1] or ""}]
    return msgs


def responder(msgs, destilado, max_tokens=MAX_TOKENS):
    """Genera en un hilo aparte y entrega el texto a medida que sale (streaming)."""
    ids = tok(plantilla(tok, msgs), return_tensors="pt", add_special_tokens=False).to(device)
    streamer = TextIteratorStreamer(tok, skip_prompt=True, skip_special_tokens=True)

    def correr():
        with GPU, version(alumno, destilado), autocast(), torch.no_grad():
            alumno.generate(**ids, max_new_tokens=max_tokens, do_sample=False, streamer=streamer,
                            pad_token_id=tok.pad_token_id)

    threading.Thread(target=correr, daemon=True).start()
    t0, primero, texto = time.time(), None, ""
    for trozo in streamer:
        primero = primero or time.time() - t0
        texto += trozo
        yield texto, ""
    n = len(tok(texto, add_special_tokens=False)["input_ids"])
    dt = time.time() - t0
    yield texto, f"{n} tokens · primer token en {1000 * (primero or dt):.0f} ms · {n / max(dt - (primero or 0), 1e-6):.0f} tokens/s"


def jugar(numeros, objetivo):
    p = {"nums": [int(n) for n in re.findall(r"\d+", numeros)], "target": int(objetivo)}
    msgs = mensajes(p)
    orig, m_orig = "", ""
    for orig, m_orig in responder(msgs, False):
        yield orig, m_orig, "", ""
    nota = verificar(orig, p)
    m_orig += " · " + ("correcta" if nota["correcta"] else "incorrecta")
    for dest, m_dest in responder(msgs, True):
        yield orig, m_orig, dest, m_dest
    nota = verificar(dest, p)
    yield orig, m_orig, dest, m_dest + " · " + ("correcta" if nota["correcta"] else "incorrecta")


def chatear(mensaje, historial):
    for texto, _ in responder(a_mensajes(historial) + [{"role": "user", "content": mensaje}], True):
        yield texto


with gr.Blocks(title="Alumno destilado") as app:
    gr.Markdown("### Qwen3-0.6B: antes y después de destilar")
    with gr.Tab("Countdown"):
        with gr.Row():
            numeros = gr.Textbox(label="Números (3, separados por coma)", value="25, 4, 9")
            objetivo = gr.Number(label="Objetivo", value=91, precision=0)
        boton = gr.Button("Resolver con los dos")
        with gr.Row():
            with gr.Column():
                gr.Markdown("**Alumno original**")
                salida_orig = gr.Textbox(lines=14, show_label=False)
                metricas_orig = gr.Markdown()
            with gr.Column():
                gr.Markdown("**Alumno destilado**")
                salida_dest = gr.Textbox(lines=14, show_label=False)
                metricas_dest = gr.Markdown()
        boton.click(jugar, [numeros, objetivo], [salida_orig, metricas_orig, salida_dest, metricas_dest])
    with gr.Tab("Chat"):
        gr.ChatInterface(chatear)

app.queue().launch(share=True, debug=False)

### Observaciones
- ¿En qué problemas se nota más la diferencia? ¿Qué hace cada uno con un problema imposible?
- ¿Qué pasó en el chat con preguntas que no son de Countdown? Destilar con un solo tipo de datos **especializa**:
  el alumno gana en lo que vio y puede perder en lo demás.
- Miren el tiempo hasta el primer token y los tokens/s. ¿Se siente como un chatbot comercial?

## 10. SECCIÓN OPCIONAL: Una API compatible con OpenAI

Ollama, vLLM y casi todos los proveedores exponen la misma API: `POST /v1/chat/completions`. Si nuestro modelo la
habla, cualquier programa escrito para OpenAI lo puede usar **cambiando solo `base_url`**. En unas 20 líneas:

In [ ]:
#@title Levantar la API (ejecutar)
import uvicorn
from fastapi import FastAPI

api = FastAPI()


@api.post("/v1/chat/completions")
def chat_completions(body: dict):
    texto = ""
    for texto, _ in responder(body["messages"], True, body.get("max_tokens") or MAX_TOKENS):
        pass
    n = len(tok(texto, add_special_tokens=False)["input_ids"])
    return {"id": "chatcmpl-lab", "object": "chat.completion", "created": int(time.time()),
            "model": body.get("model", "alumno-destilado"),
            "choices": [{"index": 0, "message": {"role": "assistant", "content": texto}, "finish_reason": "stop"}],
            "usage": {"prompt_tokens": 0, "completion_tokens": n, "total_tokens": n}}


if "SERVIDOR" not in globals():
    SERVIDOR = uvicorn.Server(uvicorn.Config(api, host="127.0.0.1", port=8000, log_level="warning"))
    threading.Thread(target=SERVIDOR.run, daemon=True).start()
    time.sleep(2)
print("API escuchando en http://127.0.0.1:8000/v1")

In [ ]:
#@title Perillas: usar la API con el cliente de OpenAI
numeros_api = "75, 3, 2"  #@param {type:"string"}
objetivo_api = 147  #@param {type:"integer"}
!pip install -q openai
from openai import OpenAI

cliente = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="no-hace-falta")
p_api = {"nums": [int(n) for n in re.findall(r"\d+", numeros_api)], "target": int(objetivo_api)}
r = cliente.chat.completions.create(model="alumno-destilado", messages=mensajes(p_api))
print(r.choices[0].message.content)
print("\nTokens:", r.usage.completion_tokens, "·", "correcta" if verificar(r.choices[0].message.content, p_api)["correcta"] else "incorrecta")

### Observaciones
- El mismo código funciona con OpenAI, Gemini, Ollama o vLLM cambiando `base_url` y `model`. ¿Qué ventaja tiene que
  todos hablen la misma API?
- Esta API atiende **de a una** pregunta. ¿Qué le faltaría para atender a cien usuarios? (Pista: sección 8.1.)

---
## Conclusiones
- Un modelo chico puede **comportarse como uno más grande** en una tarea acotada si aprende de sus respuestas.
- Lo que se destila no es solo la respuesta: es **la forma de razonar**. Sin el razonamiento, el alumno aprende el
  formato pero no a resolver.
- Copiar las **probabilidades** entrega más información por ejemplo que copiar el texto, pero exige tener al
  profesor cargado.
- Lo que se entrena con LoRA pesa megabytes: un modelo base puede tener muchos adaptadores especializados.
- Servir bien es **juntar usuarios**: el decode está limitado por memoria, así que cada pasada debe atender a muchos.

## Para seguir
- Hinton et al. (2015), *Distilling the Knowledge in a Neural Network*: https://arxiv.org/abs/1503.02531
- DeepSeek-AI (2025), *DeepSeek-R1* (sección de destilación): https://arxiv.org/abs/2501.12948
- Hugging Face H4, *Unlocking On-Policy Distillation for Any Model Family* (Countdown):
  https://huggingfaceh4-on-policy-distillation.hf.space/
- S. Paniego, *Distillation in 2026 (so far)*: https://huggingface.co/blog/sergiopaniego/distillation-2026
- TRL, `DistillationTrainer` (destilación on-policy): https://huggingface.co/docs/trl/distillation_trainer